# Text Classification: From Bag-of-Words to BERT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/text-classification.ipynb)

An educational exploration of sentiment analysis using progressively sophisticated techniques.

In [ ]:
# Configuration dictionary
# All hyperparameters and settings for the notebook are defined here
CONFIG = {
    # TF-IDF Vectorizer settings
    'tfidf_max_features': 10000,  # Keep top 10K words by frequency
    'tfidf_min_df': 5,             # Ignore words appearing in < 5 documents
    'tfidf_max_df': 0.8,           # Ignore words appearing in > 80% of documents
    'tfidf_ngram_range': (1, 2),   # Use both unigrams and bigrams
    
    # Logistic Regression settings
    'logreg_C': 1.0,               # Regularization strength (smaller = stronger)
    'logreg_max_iter': 1000,       # Maximum iterations
    
    # Vocabulary settings
    'vocab_max_size': 20000,       # Maximum vocabulary size
    
    # Dataset settings
    'max_length': 256,             # Maximum sequence length
    'batch_size': 64,              # Batch size for DataLoader
    
    # LSTM Model architecture
    'lstm_embedding_dim': 128,     # Embedding dimension
    'lstm_hidden_dim': 256,        # Hidden dimension
    'lstm_num_layers': 2,          # Number of LSTM layers
    'lstm_dropout': 0.5,           # Dropout probability
    
    # LSTM Training settings
    'lstm_learning_rate': 0.001,   # Learning rate
    'lstm_num_epochs': 1,          # Number of training epochs
    'lstm_gradient_clip': 1.0,     # Gradient clipping max norm
    
    # BERT Model settings
    'bert_model_name': 'distilbert-base-uncased',  # Pre-trained model
    'bert_train_subset_size': 5000,   # Training subset size (for speed)
    'bert_test_subset_size': 1000,    # Test subset size (for speed)
    'bert_max_length': 256,           # Maximum sequence length
    'bert_train_batch_size': 16,      # Training batch size
    'bert_eval_batch_size': 32,       # Evaluation batch size
    'bert_learning_rate': 2e-5,       # Learning rate
    'bert_weight_decay': 0.01,        # Weight decay for regularization
    'bert_num_epochs': 1,             # Number of training epochs
    'bert_logging_steps': 100,        # Logging frequency
    
    # Visualization settings
    'top_k_features': 15,          # Number of top features to visualize
    
    # Prediction settings
    'prediction_max_length': 256,  # Max length for predictions
}

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Part 1: Introduction and Motivation

### What is Text Classification?

Text classification assigns predefined categories to text documents. Sentiment analysis is a common application where we determine if text expresses positive or negative opinions.

### Learning Objectives

By the end of this notebook, you will understand:
1. **Classical ML approach**: Bag-of-words representation with logistic regression
2. **Deep learning approach**: RNN/LSTM models that capture sequential patterns
3. **Transfer learning approach**: Fine-tuning pre-trained BERT models
4. **Evaluation**: Metrics, confusion matrices, and error analysis

### Why This Progression?

Starting simple helps us appreciate why more complex models exist:
- **Bag-of-words**: Fast, interpretable, but ignores word order
- **RNNs**: Capture sequences, but struggle with long-range dependencies
- **BERT**: Pre-trained on massive text, captures bidirectional context

In [ ]:
# Environment setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import re
from typing import List, Tuple, Dict
from tqdm.auto import tqdm
from aiml_notebooks import get_device, set_seed

# Set random seeds for reproducibility
set_seed(42)

# Device configuration (safe mode for Transformer compatibility)
device = get_device(prefer_cpu=True)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## Part 2: Dataset Loading and Exploration

### The IMDb Movie Review Dataset

We'll use the IMDb dataset containing 50,000 movie reviews:
- **25,000 training reviews** (12,500 positive, 12,500 negative)
- **25,000 test reviews** (balanced)
- Reviews are labeled as positive (rating ≥ 7/10) or negative (rating ≤ 4/10)

This dataset is ideal for learning because:
1. Large enough to train deep models
2. Balanced classes (no bias toward positive/negative)
3. Real-world text with varied vocabulary and sentence structures

In [ ]:
# Download and load IMDb dataset using datasets library
from datasets import load_dataset

print("Loading IMDb dataset...")
dataset = load_dataset("imdb")

# Extract train and test sets
train_texts = dataset['train']['text']
train_labels = dataset['train']['label']
test_texts = dataset['test']['text']
test_labels = dataset['test']['label']

print(f"Training samples: {len(train_texts)}")
print(f"Test samples: {len(test_texts)}")
print(f"\nLabel distribution in training set:")
print(f"  Negative (0): {train_labels.count(0)}")
print(f"  Positive (1): {train_labels.count(1)}")

Loading IMDb dataset...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

  2025-10-27T16:15:47.971191Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x104929990>), traceback: Some(<traceback object at 0x1657e6540>) }, caller: "src/progress_update.rs:313"
    at /Users/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28



### Exploring the Data

Let's examine some samples to understand what we're working with.

In [ ]:
# Display sample reviews
print("="*80)
print("POSITIVE REVIEW EXAMPLE")
print("="*80)
pos_idx = train_labels.index(1)
print(train_texts[pos_idx][:500] + "...\n")

print("="*80)
print("NEGATIVE REVIEW EXAMPLE")
print("="*80)
neg_idx = train_labels.index(0)
print(train_texts[neg_idx][:500] + "...")

Execute the following code:

In [ ]:
# Analyze text lengths
train_lengths = [len(text.split()) for text in train_texts]
test_lengths = [len(text.split()) for text in test_texts]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_lengths, bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(np.mean(train_lengths), color='red', linestyle='--', 
                label=f'Mean: {np.mean(train_lengths):.0f} words')
axes[0].axvline(np.median(train_lengths), color='green', linestyle='--',
                label=f'Median: {np.median(train_lengths):.0f} words')
axes[0].set_xlabel('Review Length (words)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Training Set: Review Length Distribution')
axes[0].legend()

axes[1].boxplot([train_lengths, test_lengths], labels=['Train', 'Test'])
axes[1].set_ylabel('Review Length (words)')
axes[1].set_title('Review Length Comparison')

plt.tight_layout()
plt.show()

print(f"Length statistics (training):")
print(f"  Min: {min(train_lengths)} words")
print(f"  Max: {max(train_lengths)} words")
print(f"  Mean: {np.mean(train_lengths):.1f} words")
print(f"  Median: {np.median(train_lengths):.1f} words")

### Reflection Questions

1. Why is it important to have balanced classes in classification tasks?
2. How might the wide variation in review lengths affect our models?
3. What preprocessing steps might be beneficial for this text data?

## Part 3: Text Preprocessing

### Why Preprocess?

Raw text contains noise that can hinder learning:
- **HTML tags**: `<br />` in IMDb reviews
- **Case variations**: "Good" vs "good" should be treated as the same word
- **Punctuation**: May or may not carry sentiment information
- **Special characters**: URLs, numbers, symbols

### Preprocessing Strategy

We'll create a flexible preprocessing pipeline that can be adjusted for different models.

In [ ]:
def preprocess_text(text: str, 
                   lowercase: bool = True,
                   remove_html: bool = True,
                   remove_punctuation: bool = False) -> str:
    """
    Preprocess text with configurable options.
    
    Args:
        text: Input text
        lowercase: Convert to lowercase
        remove_html: Remove HTML tags
        remove_punctuation: Remove punctuation marks
    
    Returns:
        Preprocessed text
    """
    # Remove HTML tags
    if remove_html:
        text = re.sub(r'<[^>]+>', ' ', text)
    
    # Convert to lowercase
    if lowercase:
        text = text.lower()
    
    # Remove punctuation (keep spaces)
    if remove_punctuation:
        text = re.sub(r'[^\w\s]', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Test preprocessing
sample = train_texts[0][:200]
print("ORIGINAL:")
print(sample)
print("\nPREPROCESSED (minimal):")
print(preprocess_text(sample, remove_punctuation=False))
print("\nPREPROCESSED (aggressive):")
print(preprocess_text(sample, remove_punctuation=True))

Execute the following code:

In [ ]:
# Preprocess all texts for classical ML (aggressive preprocessing)
print("Preprocessing texts for classical ML models...")
train_texts_clean = [preprocess_text(t, remove_punctuation=True) for t in tqdm(train_texts)]
test_texts_clean = [preprocess_text(t, remove_punctuation=True) for t in tqdm(test_texts)]

# For deep learning, we'll use minimal preprocessing (keep punctuation for context)
train_texts_minimal = [preprocess_text(t, remove_punctuation=False) for t in train_texts]
test_texts_minimal = [preprocess_text(t, remove_punctuation=False) for t in test_texts]

print("Preprocessing complete!")

## Part 4: Baseline Model - Bag of Words + Logistic Regression

### Theory: Bag-of-Words Representation

The bag-of-words (BoW) model represents text as a vector of word counts:

**Example:**
- Vocabulary: `["good", "bad", "movie", "boring"]`
- Text: "good movie good"
- Vector: `[2, 0, 1, 0]`

**Advantages:**
- Simple and fast
- Interpretable (can see which words matter)
- Works well for short texts

**Disadvantages:**
- Ignores word order ("not good" = "good not")
- High dimensionality (vocabulary size)
- Sparse vectors (most elements are zero)

### TF-IDF Enhancement

Term Frequency-Inverse Document Frequency (TF-IDF) improves BoW by:
- **TF**: Weighing frequent words in a document
- **IDF**: Down-weighing common words across all documents (e.g., "the", "a")

Formula: `TF-IDF(word, doc) = TF(word, doc) × log(N / DF(word))`

Where:
- `TF(word, doc)` = frequency of word in document
- `N` = total number of documents
- `DF(word)` = number of documents containing word

In [ ]:
# Create TF-IDF vectorizer
print("Creating TF-IDF features...")
vectorizer = TfidfVectorizer(
    max_features=CONFIG['tfidf_max_features'],
    min_df=CONFIG['tfidf_min_df'],
    max_df=CONFIG['tfidf_max_df'],
    ngram_range=CONFIG['tfidf_ngram_range']
)

# Fit on training data and transform both train and test
X_train_tfidf = vectorizer.fit_transform(train_texts_clean)
X_test_tfidf = vectorizer.transform(test_texts_clean)

print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"Sparsity: {100 * (1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])):.2f}%")
print(f"Average non-zero elements per review: {X_train_tfidf.nnz / X_train_tfidf.shape[0]:.1f}")

### Training Logistic Regression

Logistic regression is a linear classifier that learns feature weights:

`P(positive) = σ(w₁x₁ + w₂x₂ + ... + wₙxₙ + b)`

Where:
- `σ` is the sigmoid function
- `wᵢ` are learned weights for each feature
- `xᵢ` are TF-IDF values
- `b` is the bias term

Positive weights indicate words associated with positive sentiment.

In [ ]:
# Train logistic regression
print("Training logistic regression...")
lr_model = LogisticRegression(
    max_iter=CONFIG['logreg_max_iter'],
    C=CONFIG['logreg_C'],
    random_state=42,
    verbose=1
)
lr_model.fit(X_train_tfidf, train_labels)

# Make predictions
train_preds = lr_model.predict(X_train_tfidf)
test_preds = lr_model.predict(X_test_tfidf)

# Calculate metrics
train_acc = accuracy_score(train_labels, train_preds)
test_acc = accuracy_score(test_labels, test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='binary')

print(f"\nResults:")
print(f"  Training Accuracy: {train_acc:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")
print(f"  Test Precision: {precision:.4f}")
print(f"  Test Recall: {recall:.4f}")
print(f"  Test F1-Score: {f1:.4f}")

Execute the following code:

In [ ]:
# Visualize most important features
feature_names = vectorizer.get_feature_names_out()
coefficients = lr_model.coef_[0]

# Get top positive and negative features
top_k = CONFIG['top_k_features']
top_positive_idx = np.argsort(coefficients)[-top_k:]
top_negative_idx = np.argsort(coefficients)[:top_k]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Positive features
axes[0].barh(range(top_k), coefficients[top_positive_idx])
axes[0].set_yticks(range(top_k))
axes[0].set_yticklabels([feature_names[i] for i in top_positive_idx])
axes[0].set_xlabel('Weight')
axes[0].set_title('Top Positive Sentiment Features')

# Negative features
axes[1].barh(range(top_k), coefficients[top_negative_idx])
axes[1].set_yticks(range(top_k))
axes[1].set_yticklabels([feature_names[i] for i in top_negative_idx])
axes[1].set_xlabel('Weight')
axes[1].set_title('Top Negative Sentiment Features')

plt.tight_layout()
plt.show()

### Confusion Matrix

A confusion matrix shows where our model makes mistakes:

```
                Predicted
              Neg    Pos
Actual  Neg   TN     FP
        Pos   FN     TP
```

- **True Negatives (TN)**: Correctly identified negative reviews
- **False Positives (FP)**: Negative reviews classified as positive
- **False Negatives (FN)**: Positive reviews classified as negative
- **True Positives (TP)**: Correctly identified positive reviews

In [ ]:
# Compute and visualize confusion matrix
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

print(f"\nConfusion Matrix Analysis:")
print(f"  True Negatives: {cm[0, 0]}")
print(f"  False Positives: {cm[0, 1]}")
print(f"  False Negatives: {cm[1, 0]}")
print(f"  True Positives: {cm[1, 1]}")

### Reflection Questions

1. Why do bigrams (2-word phrases) improve performance over unigrams alone?
2. What types of reviews do you think the baseline model struggles with most?
3. How might the bag-of-words assumption hurt us? (Think about "not good" vs "good")

## Part 5: Deep Learning Preparation - Custom Dataset

### Why Deep Learning?

Neural networks can learn better representations than bag-of-words:
1. **Word embeddings**: Dense vectors that capture semantic meaning
2. **Sequential modeling**: RNNs/LSTMs process word order
3. **Non-linear transformations**: Learn complex patterns

### Building a Vocabulary

We need to convert words to integer indices for embedding layers.

In [ ]:
# Build vocabulary
vocab = Vocabulary(max_size=CONFIG['vocab_max_size'])
vocab.build_vocab(train_texts_minimal)

Execute the following code:

In [ ]:
# Examine most common words
print("Most common words:")
for word, count in vocab.word_counts.most_common(20):
    print(f"  {word:15s}: {count:6d}")

### Creating PyTorch Dataset

PyTorch datasets handle data loading and batching efficiently.

In [ ]:
# Create datasets
train_dataset = SentimentDataset(train_texts_minimal, train_labels, vocab, max_length=CONFIG['max_length'])
test_dataset = SentimentDataset(test_texts_minimal, test_labels, vocab, max_length=CONFIG['max_length'])

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

# Examine a batch
batch = next(iter(train_loader))
print(f"\nBatch shape: {batch['input_ids'].shape}")
print(f"Labels shape: {batch['label'].shape}")

## Part 6: LSTM Classifier

### Theory: LSTM Networks

Long Short-Term Memory (LSTM) networks are a type of RNN designed to capture long-range dependencies.

**Architecture:**
1. **Embedding layer**: Maps word indices to dense vectors
2. **LSTM layer**: Processes sequence, maintaining hidden state
3. **Pooling**: Aggregate sequence (e.g., take last hidden state or mean)
4. **Classifier**: Linear layer(s) for final prediction

**Key LSTM Features:**
- **Forget gate**: Decides what information to discard
- **Input gate**: Decides what new information to store
- **Output gate**: Decides what to output
- **Cell state**: Long-term memory that flows through the sequence

**Advantages over Bag-of-Words:**
- Captures word order and context
- Learns semantic embeddings
- Can model negation ("not good" ≠ "good")

In [ ]:
# Create model
lstm_model = LSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=CONFIG['lstm_embedding_dim'],
    hidden_dim=CONFIG['lstm_hidden_dim'],
    num_layers=CONFIG['lstm_num_layers'],
    dropout=CONFIG['lstm_dropout']
).to(device)

# Count parameters
total_params = sum(p.numel() for p in lstm_model.parameters())
trainable_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

### Training Loop

We'll train the LSTM using:
- **Cross-entropy loss**: Standard for classification
- **Adam optimizer**: Adaptive learning rates
- **Gradient clipping**: Prevents exploding gradients in RNNs

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CONFIG['lstm_gradient_clip'])
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    """Evaluate model on validation/test set."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(loader), correct / total, all_preds, all_labels

Execute the following code:

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=CONFIG['lstm_learning_rate'])

# Training loop
num_epochs = CONFIG['lstm_num_epochs']
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("Training LSTM Classifier...\n")
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    
    # Train
    train_loss, train_acc = train_epoch(lstm_model, train_loader, optimizer, criterion, device)
    
    # Evaluate
    test_loss, test_acc, test_preds, test_labels_np = evaluate(lstm_model, test_loader, criterion, device)
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}\n")

print("Training complete!")

Execute the following code:

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train', marker='o')
axes[0].plot(history['test_loss'], label='Test', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Test Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', marker='o')
axes[1].plot(history['test_acc'], label='Test', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Test Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

Execute the following code:

In [ ]:
# Confusion matrix for LSTM
cm_lstm = confusion_matrix(test_labels_np, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - LSTM Classifier')
plt.show()

# Calculate metrics
precision, recall, f1, _ = precision_recall_fscore_support(test_labels_np, test_preds, average='binary')
print(f"LSTM Classifier Results:")
print(f"  Test Accuracy: {test_acc:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")

### Reflection Questions

1. How does LSTM accuracy compare to logistic regression?
2. Why use bidirectional LSTMs instead of unidirectional?
3. What are the tradeoffs between LSTM and simpler models? (training time, interpretability, performance)

## Part 7: Transfer Learning with BERT

### Theory: BERT (Bidirectional Encoder Representations from Transformers)

BERT is a pre-trained transformer model that revolutionized NLP:

**Key Innovations:**
1. **Bidirectional context**: Processes text in both directions simultaneously
2. **Pre-training**: Trained on massive text corpora (Wikipedia, books)
3. **Attention mechanism**: Learns which words are relevant for each word
4. **Transfer learning**: Fine-tune pre-trained model on specific tasks

**Architecture:**
- **Input**: Tokenized text with special tokens `[CLS]` (classification) and `[SEP]` (separator)
- **Transformer layers**: 12 layers (base) or 24 layers (large) of self-attention
- **Output**: Contextualized embeddings for each token
- **Classification**: Use `[CLS]` token embedding for sequence classification

**Why BERT Works:**
- Pre-trained on billions of words
- Learns general language understanding
- Fine-tuning adapts to specific tasks with little data
- Attention captures long-range dependencies

In [ ]:
# Install transformers if needed
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
except ImportError:
    print("Installing transformers library...")
    import sys
    !{sys.executable} -m pip install transformers -q
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

print("Transformers library loaded successfully!")

### Loading Pre-trained BERT

We'll use DistilBERT, a smaller, faster version of BERT:
- 66M parameters (vs 110M for BERT-base)
- 97% of BERT's performance
- 60% faster training

In [ ]:
# Load tokenizer and model
model_name = CONFIG['bert_model_name']
print(f"Loading {model_name}...")

bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # Binary classification
).to(device)

print(f"Model loaded with {sum(p.numel() for p in bert_model.parameters()):,} parameters")

### Tokenizing with BERT Tokenizer

BERT uses WordPiece tokenization:
- Splits unknown words into subwords
- Example: "unforgettable" → "un", "##forget", "##table"
- Balances vocabulary size and coverage

In [ ]:
# For demonstration, use a subset (BERT is slow to train)
# In practice, you'd use the full dataset
train_subset_size = CONFIG['bert_train_subset_size']
test_subset_size = CONFIG['bert_test_subset_size']

print(f"Creating BERT datasets (using subsets for speed)...")
bert_train_dataset = BERTDataset(
    train_texts_minimal[:train_subset_size],
    train_labels[:train_subset_size],
    bert_tokenizer,
    max_length=CONFIG['bert_max_length']
)

bert_test_dataset = BERTDataset(
    test_texts_minimal[:test_subset_size],
    test_labels[:test_subset_size],
    bert_tokenizer,
    max_length=CONFIG['bert_max_length']
)

print(f"Train samples: {len(bert_train_dataset)}")
print(f"Test samples: {len(bert_test_dataset)}")

### Fine-tuning BERT

Fine-tuning strategy:
1. **Freeze pre-trained layers (optional)**: Keep BERT weights fixed, only train classifier
2. **Small learning rate**: Pre-trained weights are already good
3. **Few epochs**: Risk of overfitting with too much training

We'll use the Hugging Face Trainer API for convenient training.

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir='./tmp/bert-finetuned',
    num_train_epochs=CONFIG['bert_num_epochs'],
    per_device_train_batch_size=CONFIG['bert_train_batch_size'],
    per_device_eval_batch_size=CONFIG['bert_eval_batch_size'],
    learning_rate=CONFIG['bert_learning_rate'],
    weight_decay=CONFIG['bert_weight_decay'],
    logging_steps=CONFIG['bert_logging_steps'],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    use_mps_device=(device.type == 'mps'),
    no_cuda=(device.type == 'mps')  # MPS is not CUDA
)

# Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Create trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train_dataset,
    eval_dataset=bert_test_dataset,
    compute_metrics=compute_metrics
)

print("Starting BERT fine-tuning...\n")
trainer.train()
print("\nFine-tuning complete!")

Execute the following code:

In [ ]:
# Evaluate on test set
print("Evaluating fine-tuned BERT...")
results = trainer.evaluate()

print(f"\nBERT Results:")
print(f"  Test Accuracy: {results['eval_accuracy']:.4f}")
print(f"  Precision: {results['eval_precision']:.4f}")
print(f"  Recall: {results['eval_recall']:.4f}")
print(f"  F1-Score: {results['eval_f1']:.4f}")

Execute the following code:

In [ ]:
# Get predictions for confusion matrix
predictions = trainer.predict(bert_test_dataset)
bert_preds = np.argmax(predictions.predictions, axis=1)
bert_labels = predictions.label_ids

# Confusion matrix
cm_bert = confusion_matrix(bert_labels, bert_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Fine-tuned BERT')
plt.show()

## Part 8: Model Comparison

Let's compare all three approaches across multiple dimensions.

In [ ]:
# Comparison table
comparison_data = {
    'Model': ['Logistic Regression', 'LSTM', 'BERT'],
    'Test Accuracy': [
        test_acc,  # From logistic regression
        history['test_acc'][-1],  # From LSTM
        results['eval_accuracy']  # From BERT
    ],
    'Parameters': [
        '~200K',  # TF-IDF features × 2 classes
        f'{trainable_params:,}',
        '66M'
    ],
    'Training Time': [
        'Seconds',
        'Minutes',
        'Hours'
    ],
    'Interpretability': [
        'High',
        'Medium',
        'Low'
    ]
}

import pandas as pd
comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

Execute the following code:

In [ ]:
# Visualize accuracy comparison
models = ['Logistic\nRegression', 'LSTM', 'BERT']
accuracies = [
    test_acc,
    history['test_acc'][-1],
    results['eval_accuracy']
]

plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracies, color=['#3498db', '#2ecc71', '#9b59b6'], alpha=0.8, edgecolor='black')
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.ylim([0.8, 1.0])
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{acc:.4f}',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### Key Takeaways

**Logistic Regression + TF-IDF:**
- Pros: Fast, interpretable, good baseline
- Cons: Ignores word order, limited by bag-of-words representation
- Best for: Quick prototypes, interpretable models, limited compute

**LSTM:**
- Pros: Captures sequential patterns, learns embeddings, reasonably fast
- Cons: Requires more data, harder to interpret, struggles with very long sequences
- Best for: Medium-sized datasets, when word order matters, custom architectures

**BERT:**
- Pros: State-of-the-art performance, pre-trained knowledge, handles context well
- Cons: Slow, resource-intensive, requires GPU, less interpretable
- Best for: High-accuracy requirements, sufficient compute, transfer learning

## Part 9: Sample Predictions and Error Analysis

Understanding where models fail helps us improve them.

In [ ]:
# Function to make predictions with LSTM
def predict_sentiment_lstm(text: str, model, vocab, device) -> Tuple[int, float]:
    """Predict sentiment with LSTM model."""
    model.eval()
    
    # Preprocess and encode
    text_clean = preprocess_text(text, remove_punctuation=False)
    indices = vocab.encode(text_clean)
    
    # Pad/truncate
    if len(indices) > 256:
        indices = indices[:256]
    else:
        indices = indices + [0] * (256 - len(indices))
    
    # Convert to tensor
    input_ids = torch.tensor([indices], dtype=torch.long).to(device)
    
    # Predict
    with torch.no_grad():
        logits = model(input_ids)
        probs = F.softmax(logits, dim=1)
        pred = logits.argmax(dim=1).item()
        confidence = probs[0, pred].item()
    
    return pred, confidence

# Function to make predictions with BERT
def predict_sentiment_bert(text: str, model, tokenizer, device) -> Tuple[int, float]:
    """Predict sentiment with BERT model."""
    model.eval()
    
    # Tokenize
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors='pt'
    )
    
    # Move to device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        pred = logits.argmax(dim=1).item()
        confidence = probs[0, pred].item()
    
    return pred, confidence

Execute the following code:

In [ ]:
# Test on custom examples
test_examples = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Terrible film. Waste of time and money.",
    "The acting was good, but the plot was confusing and boring.",
    "Not great, not terrible. Just okay.",
    "I didn't hate it, but I wouldn't recommend it either."
]

print("="*100)
print("SAMPLE PREDICTIONS")
print("="*100)

for i, text in enumerate(test_examples, 1):
    print(f"\nExample {i}: {text}")
    print("-" * 100)
    
    # LSTM prediction
    lstm_pred, lstm_conf = predict_sentiment_lstm(text, lstm_model, vocab, device)
    lstm_label = "Positive" if lstm_pred == 1 else "Negative"
    print(f"LSTM:     {lstm_label:8s} (confidence: {lstm_conf:.3f})")
    
    # BERT prediction
    bert_pred, bert_conf = predict_sentiment_bert(text, bert_model, bert_tokenizer, device)
    bert_label = "Positive" if bert_pred == 1 else "Negative"
    print(f"BERT:     {bert_label:8s} (confidence: {bert_conf:.3f})")

### Reflection Questions

1. Which model handles mixed sentiment better? (e.g., "acting was good, but plot was boring")
2. How do confidence scores differ between LSTM and BERT?
3. What types of reviews do you think would be hardest to classify correctly?

## Part 10: Conclusion and Next Steps

### What We Learned

1. **Text representation**: From sparse bag-of-words to dense embeddings
2. **Model progression**: Simple → sequential → pre-trained
3. **Tradeoffs**: Accuracy vs speed vs interpretability vs resource requirements
4. **Evaluation**: Multiple metrics and confusion matrices reveal model behavior

### Key Concepts

- **TF-IDF**: Weighs words by importance (frequent in document, rare globally)
- **Embeddings**: Dense vectors capturing semantic meaning
- **LSTMs**: Sequential models with memory for context
- **Transfer learning**: Leverage pre-trained models for new tasks
- **Fine-tuning**: Adapt pre-trained weights to specific domains

### Further Exploration

1. **Hyperparameter tuning**: Optimize learning rates, hidden sizes, dropout
2. **Data augmentation**: Back-translation, synonym replacement, paraphrasing
3. **Ensemble methods**: Combine multiple models for better predictions
4. **Attention visualization**: Understand what BERT focuses on
5. **Domain adaptation**: Fine-tune on specific domains (e.g., product reviews)
6. **Multi-class classification**: Extend to 3+ sentiment categories
7. **Aspect-based sentiment**: Identify sentiment toward specific aspects (acting, plot, etc.)

### Recommended Resources

- **Papers**:
  - BERT: "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding" (Devlin et al., 2018)
  - LSTM: "Long Short-Term Memory" (Hochreiter & Schmidhuber, 1997)
  
- **Tutorials**:
  - Hugging Face course: https://huggingface.co/course
  - Fast.ai NLP: https://www.fast.ai/
  
- **Datasets**:
  - SST-2 (Stanford Sentiment Treebank)
  - Yelp reviews
  - Amazon product reviews

## Experiments to Try

1. **Adjust preprocessing**: Try keeping/removing punctuation for each model
2. **Modify architectures**: Add more LSTM layers, change hidden dimensions
3. **Different BERT models**: Try `bert-base-uncased`, `roberta-base`, or domain-specific BERT
4. **Class imbalance**: Artificially imbalance the dataset and use weighted loss
5. **Sequence length**: Vary max_length and observe effects on performance
6. **Learning rate schedules**: Try warmup and decay for BERT fine-tuning
7. **Error analysis**: Find systematically misclassified examples and understand why